In [17]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [ ]:
#!pip install torch==2.3.1+cu118 torchvision==0.18.1+cu118 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118 
# #!pip uninstall flash-attn -y 
# #!pip install "numpy<2" 
# #!pip install openpyxl 
# #!pip install SentencePiece

In [18]:
#!pip install "numpy<2"
import pandas as pd
import os
import numpy as np

from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from transformers import MarianMTModel, MarianTokenizer
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM


# 1) Preprocesamiento de los datos


## Functions

In [19]:


import pandas as pd
import re
import unicodedata


def get_expresions_to_delete():
  """
  Devuelve una lista de patrones regex para eliminar
  instrucciones o textos no deseados en el resumen.
  """
  delete = [
    r"resumen del proyecto\s*\(1\s*p[aá]gina\)",
    r"debe ser suficientemente informativo.*?proyecto",
    r"problema que se abordar[áa],\s*objetivos,\s*metodolog[ií]a y resultados que se esperan[\s\S]*?de evaluadores",
    r"problema que se abordar[áa],\s*objetivos,\s*metodolog[ií]a y resultados que se esperan[\s\S]*?investigaci[oó]n",
    r"debe considerarse que un resumen bien formulado facilita.*?evaluadores",
    r"DESCRIBE THE MAIN ISSUES TO BE ADDRESSED[\s\S]*?EXPECTED RESULTS\.",
    r"THE MAXIMUM LENGTH FOR THIS SECTION[\s\S]*?SIMILAR\).",
    r"AVOID INCLUDING IN THIS SECTION INFORMATION[\s\S]*?BACKGROUNDS\."
    ]
  return delete

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Normalizar caracteres unicode (acentos, etc.)
    text = unicodedata.normalize("NFKC", text)

    # Estandarización de carácteres de espacio
    text = re.sub(r'[\u00A0\u1680\u180E\u2000-\u200F\u202F\u205F\u3000\uFEFF]', ' ', text)
    text = re.sub(r'\_x000D_', ' ', text)

    # Eliminar referencias tipo [1], [12], etc.
    text = re.sub(r'\[\d+\]', '', text)

    # Eliminar URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Eliminar números tipo "1. Introducción"
    text = re.sub(r'\d+\.\s*[A-Za-zÁÉÍÓÚáéíóúñÑ]+', '', text)

    # Eliminar caracteres no alfanuméricos (excepto puntuación básica)
    #text = re.sub(r'[^\w\s.,;:()\[\]¿?!¡%\-\\n]', '', text)

    # Eliminar múltiples espacios
    text = re.sub(r'[ \t]+', ' ', text)

    # Eliminar instrucciones comunes del formulario
    delete = get_expresions_to_delete()
    for prhase in delete:
        text = re.sub(prhase, '', text, flags=re.IGNORECASE | re.DOTALL)

    #busca cualquier secuencia de 3 o más saltos de línea consecutivos.
    text = re.sub(r'\n{3,}', '\n', text)

    return text.strip().lower()


#Esta función aún no funciona completamente bien, así que no está en el Pipeline
def expand_acronyms(text):
    acronyms = {}
    # Encontrar definiciones de acrónimos tipo "Texto largo (ACR)"
    pattern = re.compile(r'\b([A-Z][A-Za-z0-9&.\s]+?)\s*\(\s*([A-Z]{2,})\s*\)')
    for match in pattern.finditer(text):
        long_form, short_form = match.groups()
        acronyms[short_form] = long_form.strip()

    # Eliminar la definición original dejando solo la forma larga
    text = pattern.sub(lambda m: m.group(1), text)

    # Reemplazar todas las apariciones del acrónimo por la forma larga
    if acronyms:
        acronym_pattern = re.compile(r'\b(' + '|'.join(map(re.escape, acronyms.keys())) + r')\b')
        text = acronym_pattern.sub(lambda m: acronyms[m.group(0)], text)

    return text




## Preprocess

In [ ]:

# 1) Cargar datos
path = "/tmp/data"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")


# 2) Preprocesar los datos

#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].map(clean_text)

# 2) Traducción del texto

## Functions

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import MarianMTModel, MarianTokenizer
import torch
import langid
import re
import pandas as pd

class translator():
    """
    Clase para traducir texto del español al inglés utilizando un modelo y tokenizer de Hugging Face.
    Incluye detección de idioma, segmentación en fragmentos y unión de la traducción final.
    """
    def __init__(self, model, tokenizer, max_input_tokens=512):
        """
        Inicializa el traductor cargando el modelo y tokenizer en GPU si está disponible.
        Args:
            model: Modelo de traducción.
            tokenizer: Tokenizer asociado al modelo.
            max_input_tokens (int): Máximo de tokens por fragmento.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # Cargar modelo y tokenizer
        self.tokenizer = tokenizer
        self.model = model.to(self.device)
        self.max_input_tokens = max_input_tokens

    def split_text(self, text_to_split):
        """
        Divide un texto en fragmentos manejables según el límite de tokens del modelo.
        Args:
            text_to_split (str): Texto original a dividir.
        Returns:
            list: Lista de fragmentos como objetos Document.
        """
        # Splitter basado en el tokenizador de Helsinki (cuenta tokens reales)
        text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer=self.tokenizer,
            chunk_size=self.max_input_tokens,
            chunk_overlap=0,
            separators=["\n\n", ".", ",", " "] #Jerarquía de separadores
        )

        texts = text_splitter.create_documents([text_to_split])
        return texts

    def translate_esp_en(self, text_to_split):
        """
        Traduce un texto del español al inglés dividiéndolo en fragmentos y recomponiendo el resultado.
        Args:
            text_to_split (str): Texto en español.
        Returns:
            str: Texto traducido al inglés.
        """
        #Split text
        texts = self.split_text(text_to_split)
        # Translate
        translated_chunks = []
        for chunk in texts:
            encoded = self.tokenizer(chunk.page_content, return_tensors="pt", truncation=True, max_length=512).to(self.device)
            with torch.inference_mode():
                out_ids = self.model.generate(
                    **encoded,
                    num_beams=4,
                    max_new_tokens=self.max_input_tokens,   # evita salidas cortas
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
            translated = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)[0]
            translated_chunks.append(translated)
        # Traducción final unida
        final_translation = "\n\n".join(translated_chunks)
        return final_translation

    # Detección y traducción
    def detect_and_translate(self, text):
        """
        Detecta el idioma del texto y lo traduce si está en español.
        Args:
            text (str): Texto de entrada.
        Returns:
            str: Texto traducido o el mismo texto si no es español.
        """
        lang, _ = langid.classify(text)
        if lang == 'es':
            return self.translate_esp_en(text)

        return text


def final_clean(text):
    """
    Limpia un texto eliminando saltos de línea, tabs y espacios múltiples.
    Args:
        text (str): Texto de entrada.
    Returns:
        str: Texto limpio y sin espacios innecesarios.
    """
    if not isinstance(text, str):
        return ""
    # Reemplaza saltos de línea y tabs por un espacio
    text = re.sub(r'[\r\n\t]+', ' ', text)
    # Colapsa espacios múltiples
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def gen_text_for_embedding(df, cols):
    """
    df   : DataFrame de entrada
    cols : lista de nombres de columnas a procesar y concatenar
    """
    df = df.copy()
    # Aplica limpieza a cada columna especificada
    for col in cols:
        df[col] = df[col].apply(final_clean)
    # Concatena las columnas limpias en una nueva columna
    df["text_for_embedding_translated"] = df[cols].agg(" ".join, axis=1)
    return df


## Translator

In [ ]:
#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
df[list(cols.values())] = df[list(cols.keys())].map(trans.detect_and_translate)
#Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

In [ ]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
#   para embedding.
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
df = gen_text_for_embedding(df, cols)
#Guardado de resultados
#Voy a cambiar esto por csv
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)

savepath=os.path.join(path, "data/data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()